# 19. 🧰 Прикладное: причинный вывод и эксперименты

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/19_causal_inference.ipynb)

Код из раздела [modules/19_causal_inference.md](https://github.com/justxor/math-for-ai-2026/blob/main/modules/19_causal_inference.md#m19). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### 🏋️ Практика модуля

**19.1.** Смоделируйте конфаундинг: тяжесть болезни повышает вероятность лечения и ухудшает исход; лечение улучшает исход на +1. Сравните наивную оценку и регрессию с контролем.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
rng = np.random.default_rng(0); n = 20_000
Z = rng.normal(size=n)                                    # тяжесть
T = (rng.random(n) < 1 / (1 + np.exp(-2 * Z))).astype(float)
Y = 1.0 * T - 2.0 * Z + rng.normal(size=n)                # истинный эффект +1
naive = Y[T == 1].mean() - Y[T == 0].mean()
X = np.c_[np.ones(n), T, Z]
adj = np.linalg.lstsq(X, Y, rcond=None)[0][1]
print(round(naive, 2), round(adj, 2))                     # наивно ≈ −1.4 (лечение «вредит»!), с контролем ≈ +1.0

### 🏋️ Практика модуля

**19.2.** Тот же пример: оцените эффект через IPW с известным propensity score.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
e = 1 / (1 + np.exp(-2 * Z))
ate_ipw = np.mean(T * Y / e) - np.mean((1 - T) * Y / (1 - e))
print(round(ate_ipw, 2))   # ≈ 1.0

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def simpson():
    rng = np.random.default_rng(2)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
    xs, ys = [], []
    for g, (mx, my), c in zip(range(3), [(2, 8), (5, 5), (8, 2)], C):
        x = rng.normal(mx, 0.8, 60); y = my + 0.8 * (x - mx) + rng.normal(0, 0.5, 60)
        axes[1].scatter(x, y, s=12, color=c, label=f"группа {g + 1}: наклон +0.8")
        xs.append(x); ys.append(y)
    X = np.concatenate(xs); Y = np.concatenate(ys)
    k, b = np.polyfit(X, Y, 1)
    axes[0].scatter(X, Y, s=12, color="gray")
    xx = np.linspace(0, 10, 10)
    axes[0].plot(xx, k * xx + b, color="k", lw=2, label=f"общий наклон {k:+.2f}")
    axes[0].legend(); axes[0].set_title("Без учёта групп: «чем больше x, тем меньше y»")
    for x, y, c in zip(xs, ys, C):
        kk, bb = np.polyfit(x, y, 1); axes[1].plot(xx, kk * xx + bb, color=c, lw=1.5)
    axes[1].set_ylim(-2, 12); axes[1].legend(fontsize=8)
    axes[1].set_title("Внутри каждой группы связь положительная: x ↑ ⇒ y ↑")
    fig.suptitle("Парадокс Симпсона: конфаундер (группа) переворачивает вывод", y=1.02)
    save(fig, "simpson")

simpson()